<a href="https://colab.research.google.com/github/samreenfathima18/guvi-project4/blob/main/project4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pandas scikit-learn streamlit selenium
!npm install -g localtunnel
!apt-get update -qq && apt-get install -y -qq chromium-browser chromium-chromedriver
!mkdir -p data


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 2s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋npm notice
npm notice New major version of npm available! 10.8.2 -> 12.2.3
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.2.3
npm notice To update run: npm install -g npm@12.2.3
npm notice
⠋W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../0-systemd-sysv_255.4-1ubuntu8.19_amd64.deb ...
Unpacking systemd-sysv (255.4-1ubuntu8.19) over (255.4-1ubuntu8.18) ...
Preparing to unpack .../1-systemd-dev_255.4-1ubuntu8.19_all.deb ...
Unpacking systemd-dev (255.4-1ubuntu8.19) over (255.4-1ubuntu8.18) ...
Preparing to unpack .../2-libpam-systemd_255.4-1ubuntu8.19_amd64.deb ...
Unpacking libpam-systemd:amd64 (

In [2]:
import shutil
from google.colab import files
import pandas as pd

uploaded = files.upload()  # Click 'Choose Files' and select your CSV
for filename in uploaded.keys():
    shutil.move(filename, "data/imdb2024_movies.csv")

df_check = pd.read_csv("data/imdb2024_movies.csv")
print("Dataset loaded successfully! Shape:", df_check.shape)
print("Columns found:", df_check.columns.tolist())
df_check.head(3)


Saving imdb2024_movies.csv to imdb2024_movies.csv
Dataset loaded successfully! Shape: (5099, 2)
Columns found: ['Movie Name', 'Storyline']


,Movie Name,Storyline
0,The Substance,A fading celebrity takes a black-market drug: ...
1,The Life of Chuck,"A life-affirming, genre-bending story about th..."
2,Bone Lake,A couple's vacation at a secluded estate is up...


In [3]:
%%writefile scraper.py
"""Selenium web scraper for 2024 IMDb movie titles and storylines."""

import time

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By


def scrape_imdb_movies(url: str, output_csv: str) -> pd.DataFrame:
    """Scrape 2024 movie titles and storylines from IMDb and save to CSV."""
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0")

    driver = webdriver.Chrome(options=chrome_options)
    movies = []

    try:
        driver.get(url)
        time.sleep(3)

        movie_cards = driver.find_elements(
            By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item"
        )

        for card in movie_cards:
            try:
                raw_title = card.find_element(
                    By.CSS_SELECTOR, "h3.ipc-title__text"
                ).text
                title = (
                    raw_title.split(". ", 1)
                    if ". " in raw_title
                    else raw_title
                )
                storyline = card.find_element(
                    By.CSS_SELECTOR, "div.ipc-html-content-inner-div"
                ).text
                movies.append({"Movie Name": title, "Storyline": storyline})
            except Exception:
                continue
    finally:
        driver.quit()

    df = pd.DataFrame(movies)
    df.to_csv(output_csv, index=False)
    print(f"Scraped {len(df)} movies and saved to {output_csv}")
    return df


if __name__ == "__main__":
    IMDB_2024_URL = (
        "https://imdb.com"
    )
    scrape_imdb_movies(IMDB_2024_URL, "data/scraped_imdb2024.csv")


Writing scraper.py


In [7]:
%%writefile app.py
"""Streamlit web application for IMDb 2024 Movie Recommendations."""

import re
import string

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import streamlit as st

st.set_page_config(
    page_title="IMDb 2024 Movie Recommender",
    page_icon="🎬",
    layout="wide",
)


def clean_text(text: str) -> str:
    """Lowercase text, remove punctuation, and normalize whitespace."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


@st.cache_data
def load_data(filepath: str = "data/imdb2024_movies.csv") -> pd.DataFrame:
    """Load, clean, and preprocess the IMDb 2024 movies dataset."""
    df = pd.read_csv(filepath)
    df = df.dropna(subset=["Movie Name", "Storyline"]).reset_index(drop=True)
    df["Cleaned_Storyline"] = df["Storyline"].apply(clean_text)
    return df


@st.cache_resource
def build_tfidf_model(storylines: pd.Series):
    """Fit the TF-IDF vectorizer once and cache the feature matrix."""
    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(storylines)
    return vectorizer, matrix


def get_recommendations(
    user_query: str,
    vectorizer: TfidfVectorizer,
    tfidf_matrix,
    dataframe: pd.DataFrame,
    top_n: int = 5,
) -> list[dict]:
    """Compute cosine similarity and return the top N matching movies."""
    cleaned_query = clean_text(user_query)
    query_vector = vectorizer.transform([cleaned_query])
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_indices = similarity_scores.argsort()[::-1][:top_n]

    results = []
    for idx in top_indices:
        if similarity_scores[idx] > 0:
            results.append(
                {
                    "Movie Name": dataframe.iloc[idx]["Movie Name"],
                    "Storyline": dataframe.iloc[idx]["Storyline"],
                    "Match Score": f"{similarity_scores[idx] * 100:.1f}%",
                }
            )
    return results


def main() -> None:
    """Render the interactive Streamlit application."""
    st.title("🎬 IMDb 2024 Movie Recommendation System")
    st.markdown(
        "Enter a movie storyline or plot concept below to discover the "
        "**top 5 most similar 2024 movies** using TF-IDF and Cosine Similarity."
    )

    df = load_data()
    tfidf_vectorizer, tfidf_matrix = build_tfidf_model(df["Cleaned_Storyline"])

    with st.sidebar:
        st.header("System Overview")
        st.write(f"**Movies Indexed:** {len(df)}")
        st.write("**NLP Method:** TF-IDF Vectorizer")
        st.write("**Similarity Metric:** Cosine Similarity")

    user_input = st.text_area(
        "Enter a custom storyline or plot summary:",
        placeholder=(
            "A young wizard begins his journey at a magical school where he "
            "makes friends and enemies, facing dark forces along the way."
        ),
        height=120,
    )

    if st.button("Find Similar Movies", type="primary"):
        if not user_input.strip():
            st.warning("Please enter a storyline description first.")
            return

        with st.spinner("Analyzing storyline similarity..."):
            recommendations = get_recommendations(
                user_input, tfidf_vectorizer, tfidf_matrix, df, top_n=5
            )

        if recommendations:
            st.subheader("Top 5 Recommended Movies")
            st.divider()
            for rank, movie in enumerate(recommendations, start=1):
                col1, col2 = st.columns([4, 1])
                with col1:
                    st.markdown(f"### {rank}. {movie['Movie Name']}")
                    st.write(movie["Storyline"])
                with col2:
                    st.metric(
                        label="Match Confidence",
                        value=movie["Match Score"],
                    )
                st.divider()
        else:
            st.info("No matching movies found. Try using different keywords.")


if __name__ == "__main__":
    main()


Overwriting app.py


In [8]:
import urllib.request

tunnel_ip = (
    urllib.request.urlopen("https://icanhazip.com")
    .read()
    .decode("utf8")
    .strip()
)
print("1. Copy this Tunnel Password:", tunnel_ip)
print("2. Click the 'your url is: https://...' link below and paste the password.")

!streamlit run app.py &>/content/logs.txt &
!npx localtunnel --port 8501


1. Copy this Tunnel Password: 136.108.184.244
2. Click the 'your url is: https://...' link below and paste the password.
⠙⠹⠸⠼⠴⠦your url is: https://young-maps-strive.loca.lt
^C
